In [ ]:
!pip install jams mir_eval autochord tf_keras onnxruntime
!pip install basic-pitch==0.4.0 --no-deps
!pip install pretty_midi resampy

import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Force CPU — avoids CUDA errors with Basic Pitch

import jams
import mir_eval
import librosa
import numpy as np
import pandas as pd
import autochord
from pathlib import Path

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Paths
GUITARSET_DIR = Path('/content/drive/MyDrive/Capstone/FullGuitarSetData')
AUDIO_DIR = GUITARSET_DIR / 'AudioFiles'
ANNOTATIONS_DIR = GUITARSET_DIR / 'JamsFiles'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


https://guitarset.weebly.com/

In [ ]:
# ============================================================
# Cell 2: GuitarSet Loader
# ============================================================
# Loads a GuitarSet recording's audio path and parsed ground truth
# annotations into a single structured dict for downstream evaluation.
#
# GuitarSet recordings follow a naming convention like:
#   00_BN1-129-Eb_comp
#   ^   ^   ^   ^  ^
#   |   |   |   |  └─ comp (chord backing) or solo (melodic)
#   |   |   |   └──── key
#   |   |   └──────── tempo in BPM
#   |   └──────────── style (BN, Funk, Jazz, Rock, SS)
#   └──────────────── recording ID

# Standard guitar tuning — MIDI pitch of each open string
# Index 0 = low E (string 6 in guitar notation)
# Index 5 = high E (string 1 in guitar notation)
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]


def load_guitarset_recording(recording_id, audio_dir=AUDIO_DIR, annotations_dir=ANNOTATIONS_DIR):
    """
    Load a GuitarSet recording's audio path and ground truth annotations.

    Args:
        recording_id: Filename stem without extension, e.g. "00_BN1-129-Eb_comp"
        audio_dir: Path to folder containing .wav files
        annotations_dir: Path to folder containing .jams files

    Returns:
        dict with keys:
            - id: the recording_id string
            - audio_path: full path to the .wav file
            - duration: length of the recording in seconds
            - style: e.g. "BN1", "Funk1", "Jazz2"
            - tempo: BPM as int
            - key_in_filename: key as named in the filename, e.g. "Eb"
            - is_comp: True for comping, False for solo
            - key: ground truth key as labeled in the JAMS file (e.g. "Eb:major")
            - chords: list of (start, end, label) tuples for ground truth chords
            - notes: list of dicts with keys (start, duration, string, fret, midi, note_name)
            - beats: list of beat onset times in seconds
            - jam: the raw jams object (in case you need to dig deeper)

    Raises:
        FileNotFoundError: if either the audio or annotation file is missing.
    """
    audio_path = Path(audio_dir) / f"{recording_id}_mic.wav"
    jams_path = Path(annotations_dir) / f"{recording_id}.jams"

    if not audio_path.exists():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")
    if not jams_path.exists():
        raise FileNotFoundError(f"Annotation file not found: {jams_path}")

    # Parse filename for metadata
    # Format: "00_Style-TEMPO-KEY_compORsolo"
    stem = recording_id
    parts = stem.split('_')
    # parts[0] = "00", parts[1] = "BN1-129-Eb", parts[2] = "comp" or "solo"
    style_tempo_key = parts[1].split('-')
    style = style_tempo_key[0]
    tempo = int(style_tempo_key[1])
    key_in_filename = style_tempo_key[2]
    is_comp = (parts[2] == 'comp')

    # Load the JAMS annotation
    jam = jams.load(str(jams_path))

    # --- Extract key (song-level) ---
    key_label = None
    key_anns = jam.search(namespace='key_mode')
    if key_anns and len(key_anns[0].data) > 0:
        key_label = key_anns[0].data[0].value

    # --- Extract chord progression (time-aligned) ---
    chords = []
    chord_anns = jam.search(namespace='chord')
    if chord_anns:
        for obs in chord_anns[0].data:
            chords.append((obs.time, obs.time + obs.duration, obs.value))

    # --- Extract beat onsets ---
    beats = []
    beat_anns = jam.search(namespace='beat_position')
    if beat_anns:
        beats = [obs.time for obs in beat_anns[0].data]

    # --- Extract per-string note annotations and derive fret positions ---
    notes = []
    note_anns = jam.search(namespace='note_midi')
    for string_idx, anno in enumerate(note_anns):
        for obs in anno.data:
            midi_pitch = obs.value
            fret = round(midi_pitch - OPEN_STRING_MIDI[string_idx])
            notes.append({
                'start': obs.time,
                'duration': obs.duration,
                'string': string_idx,            # 0 = low E, 5 = high E
                'fret': fret,                    # 0 = open
                'midi': round(midi_pitch),
                'note_name': _midi_to_note_name(midi_pitch),
            })
    # Sort chronologically
    notes.sort(key=lambda n: n['start'])

    return {
        'id': recording_id,
        'audio_path': str(audio_path),
        'duration': jam.file_metadata.duration,
        'style': style,
        'tempo': tempo,
        'key_in_filename': key_in_filename,
        'is_comp': is_comp,
        'key': key_label,
        'chords': chords,
        'notes': notes,
        'beats': beats,
        'jam': jam,
    }


def _midi_to_note_name(midi):
    """Convert MIDI pitch number to note name string (e.g. 64 -> 'E4')."""
    pitch_classes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    octave = int(midi) // 12 - 1
    pc = pitch_classes[int(midi) % 12]
    return f"{pc}{octave}"




In [ ]:
# ============================================================
# Cell 2: Fretboard Lookup Table
# ============================================================
# The foundational data structure. Every (string, fret) pair maps
# deterministically to a MIDI pitch, and every MIDI pitch has a
# known set of valid (string, fret) positions on the guitar.
#
# String indexing convention:
#   0 = low E (thickest string, lowest pitch) — open MIDI = 40
#   5 = high E (thinnest string, highest pitch) — open MIDI = 64
# This matches GuitarSet's convention.

# Standard tuning — open-string MIDI values
OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
MAX_FRET = 22  # most acoustic and electric guitars

# Pitch class names for display
PITCH_CLASSES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']


def fret_to_midi(string: int, fret: int) -> int:
    """Convert a (string, fret) position to its MIDI pitch.

    Example: fret_to_midi(5, 0) -> 64 (high E open)
             fret_to_midi(0, 3) -> 43 (low E string, fret 3 = G2)
    """
    if not 0 <= string <= 5:
        raise ValueError(f"String must be 0-5, got {string}")
    if not 0 <= fret <= MAX_FRET:
        raise ValueError(f"Fret must be 0-{MAX_FRET}, got {fret}")
    return OPEN_STRING_MIDI[string] + fret


def midi_to_positions(midi: int) -> list[tuple[int, int]]:
    """Return all valid (string, fret) positions that produce this MIDI pitch.

    Example: midi_to_positions(64) -> [(2, 14), (3, 9), (4, 5), (5, 0)]
        E4 can be played on the D string fret 14, G string fret 9,
        B string fret 5, or high E string open.
    """
    positions = []
    for string in range(6):
        fret = midi - OPEN_STRING_MIDI[string]
        if 0 <= fret <= MAX_FRET:
            positions.append((string, fret))
    return positions


def midi_to_note_name(midi: int) -> str:
    """Convert MIDI number to human-readable note name (e.g. 64 -> 'E4')."""
    octave = midi // 12 - 1
    pc = PITCH_CLASSES[midi % 12]
    return f"{pc}{octave}"


# Smoke test
print("Fretboard lookup test:\n")

print("Open strings (low to high):")
for s in range(6):
    midi = fret_to_midi(s, 0)
    print(f"  String {s}: MIDI {midi} = {midi_to_note_name(midi)}")

print("\nAll positions for E4 (MIDI 64):")
for s, f in midi_to_positions(64):
    print(f"  String {s} (open {midi_to_note_name(OPEN_STRING_MIDI[s])}), fret {f}")

print("\nAll positions for A2 (MIDI 45):")
for s, f in midi_to_positions(45):
    print(f"  String {s} (open {midi_to_note_name(OPEN_STRING_MIDI[s])}), fret {f}")

Fretboard lookup test:

Open strings (low to high):
  String 0: MIDI 40 = E2
  String 1: MIDI 45 = A2
  String 2: MIDI 50 = D3
  String 3: MIDI 55 = G3
  String 4: MIDI 59 = B3
  String 5: MIDI 64 = E4

All positions for E4 (MIDI 64):
  String 1 (open A2), fret 19
  String 2 (open D3), fret 14
  String 3 (open G3), fret 9
  String 4 (open B3), fret 5
  String 5 (open E4), fret 0

All positions for A2 (MIDI 45):
  String 0 (open E2), fret 5
  String 1 (open A2), fret 0


In [ ]:
# ============================================================
# Cell 3: TabNote Data Structure
# ============================================================
# The unified data structure for notes that have been assigned
# to specific fretboard positions. This matches the shape of
# GuitarSet's ground-truth annotations, so evaluation is direct.
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class TabNote:
    """A note with its assigned position on the fretboard."""
    start: float           # seconds
    duration: float        # seconds
    midi: int              # MIDI pitch (60 = middle C, 64 = E4, etc.)
    string: int            # 0 (low E) to 5 (high E)
    fret: int              # 0 (open) to MAX_FRET
    confidence: float = 1.0  # how sure the algorithm is about this assignment

    @property
    def note_name(self) -> str:
        return midi_to_note_name(self.midi)

    def __repr__(self) -> str:
        return (f"TabNote(t={self.start:.2f}s, {self.note_name}, "
                f"string={self.string}, fret={self.fret})")


# Smoke test
note = TabNote(start=1.5, duration=0.4, midi=64, string=3, fret=9)
print(f"Created: {note}")
print(f"  Pitch verification: fret_to_midi({note.string}, {note.fret}) = "
      f"{fret_to_midi(note.string, note.fret)}, matches midi={note.midi}: "
      f"{fret_to_midi(note.string, note.fret) == note.midi}")

Created: TabNote(t=1.50s, E4, string=3, fret=9)
  Pitch verification: fret_to_midi(3, 9) = 64, matches midi=64: True


In [ ]:
# ============================================================
# Cell 4: Naive Baseline Fret Assignment
# ============================================================
# For each note, pick the position with the lowest fret number.
# This is intentionally simple

def naive_assign(notes: list[dict]) -> list[TabNote]:
    """Assign fret positions using the simplest possible rule:
    pick the lowest-fret valid position for each note independently.

    Args:
        notes: list of dicts with keys 'start', 'duration', 'midi'
               (the format from Basic Pitch's note detection)

    Returns:
        list of TabNote objects with assigned positions
    """
    tab_notes = []
    for n in notes:
        positions = midi_to_positions(n['midi'])
        if not positions:
            # Note is outside guitar range — skip
            continue

        # Pick lowest-fret position. Ties broken by lowest string.
        best = min(positions, key=lambda p: (p[1], p[0]))

        tab_notes.append(TabNote(
            start=n['start'],
            duration=n['duration'],
            midi=n['midi'],
            string=best[0],
            fret=best[1],
            confidence=0.5,  # low confidence — we're just guessing
        ))
    return tab_notes


# Smoke test — make up a simple riff and see how naive assignment handles it
test_notes = [
    {'start': 0.0, 'duration': 0.5, 'midi': 64},  # E4
    {'start': 0.5, 'duration': 0.5, 'midi': 67},  # G4
    {'start': 1.0, 'duration': 0.5, 'midi': 71},  # B4
    {'start': 1.5, 'duration': 0.5, 'midi': 76},  # E5
]

naive_tab = naive_assign(test_notes)
print("Naive assignment of E4 → G4 → B4 → E5:")
for tn in naive_tab:
    print(f"  {tn}")

Naive assignment of E4 → G4 → B4 → E5:
  TabNote(t=0.00s, E4, string=5, fret=0)
  TabNote(t=0.50s, G4, string=5, fret=3)
  TabNote(t=1.00s, B4, string=5, fret=7)
  TabNote(t=1.50s, E5, string=5, fret=12)


In [ ]:
# ============================================================
# Cell 5: Music Theory Helpers
# ============================================================
# Functions that encode music theory knowledge — what notes are
# in a key, what notes are chord tones, etc.

# Scale degree patterns (semitone intervals from tonic)
MAJOR_SCALE_INTERVALS = [0, 2, 4, 5, 7, 9, 11]
MINOR_SCALE_INTERVALS = [0, 2, 3, 5, 7, 8, 10]


def note_name_to_pitch_class(note_name: str) -> int:
    """Convert note name to pitch class number (0-11).
    Handles both sharps and flats: 'C#' = 'Db' = 1
    """
    # Normalize flats to sharps
    flat_to_sharp = {'Db': 'C#', 'Eb': 'D#', 'Gb': 'F#', 'Ab': 'G#', 'Bb': 'A#'}
    note_name = flat_to_sharp.get(note_name, note_name)
    return PITCH_CLASSES.index(note_name)


def notes_in_key(tonic: str, mode: str = 'major') -> set[int]:
    """Return the set of pitch classes (0-11) in a given key.

    Example: notes_in_key('D', 'major') -> {2, 4, 6, 7, 9, 11, 1}
             (D, E, F#, G, A, B, C#)
    """
    tonic_pc = note_name_to_pitch_class(tonic)
    intervals = MAJOR_SCALE_INTERVALS if mode == 'major' else MINOR_SCALE_INTERVALS
    return {(tonic_pc + i) % 12 for i in intervals}


def is_in_key(midi: int, tonic: str, mode: str = 'major') -> bool:
    """Check if a MIDI note belongs to the given key's scale."""
    return (midi % 12) in notes_in_key(tonic, mode)


# Chord-tone definitions for the most common chord qualities
# Expressed as semitone intervals from the chord root
CHORD_INTERVALS = {
    'maj':  [0, 4, 7],          # major triad
    'min':  [0, 3, 7],          # minor triad
    '7':    [0, 4, 7, 10],      # dominant 7th
    'maj7': [0, 4, 7, 11],      # major 7th
    'min7': [0, 3, 7, 10],      # minor 7th
    'dim':  [0, 3, 6],          # diminished triad
}


def parse_chord_label(label: str) -> Optional[tuple[str, str]]:
    """Parse a chord label like 'D#:maj' or 'F#:min' into (root, quality).
    Returns None for non-chord labels like 'N'.
    """
    if not label or label == 'N':
        return None
    if ':' in label:
        root, quality = label.split(':', 1)
    else:
        # No quality specified — assume major
        root, quality = label, 'maj'
    # Normalize 'major'/'minor' to 'maj'/'min'
    quality = quality.replace('major', 'maj').replace('minor', 'min')
    return root, quality


def chord_tones(label: str) -> set[int]:
    """Return the set of pitch classes that are chord tones for this chord.

    Example: chord_tones('C:maj') -> {0, 4, 7}  (C, E, G)
             chord_tones('D:min') -> {2, 5, 9}  (D, F, A)
    """
    parsed = parse_chord_label(label)
    if parsed is None:
        return set()
    root, quality = parsed
    root_pc = note_name_to_pitch_class(root)
    intervals = CHORD_INTERVALS.get(quality, CHORD_INTERVALS['maj'])
    return {(root_pc + i) % 12 for i in intervals}


def is_chord_tone(midi: int, chord_label: str) -> bool:
    """Check if a MIDI note is a chord tone of the given chord."""
    return (midi % 12) in chord_tones(chord_label)


# Smoke test
print("Notes in D major:")
notes = sorted(notes_in_key('D', 'major'))
print(f"  Pitch classes: {notes}")
print(f"  Note names: {[PITCH_CLASSES[pc] for pc in notes]}")

print("\nChord tones of F# minor:")
tones = sorted(chord_tones('F#:min'))
print(f"  Pitch classes: {tones}")
print(f"  Note names: {[PITCH_CLASSES[pc] for pc in tones]}")

print("\nIs E4 (MIDI 64) in D major?", is_in_key(64, 'D', 'major'))
print("Is E4 a chord tone of Em?", is_chord_tone(64, 'E:min'))
print("Is F4 (MIDI 65) in D major?", is_in_key(65, 'D', 'major'))

Notes in D major:
  Pitch classes: [1, 2, 4, 6, 7, 9, 11]
  Note names: ['C#', 'D', 'E', 'F#', 'G', 'A', 'B']

Chord tones of F# minor:
  Pitch classes: [1, 6, 9]
  Note names: ['C#', 'F#', 'A']

Is E4 (MIDI 64) in D major? True
Is E4 a chord tone of Em? True
Is F4 (MIDI 65) in D major? False


In [ ]:
# ============================================================
# Cell 6: Chord Context Lookup
# ============================================================

def chord_at_time(time: float, chord_progression: list[tuple]) -> Optional[str]:
    """Find which chord (if any) is active at a given time.

    Args:
        time: time in seconds
        chord_progression: list of (start, end, label) tuples

    Returns:
        chord label string, or None if no chord is active at that time
    """
    for start, end, label in chord_progression:
        if start <= time < end:
            return label
    return None


# Smoke test
fake_progression = [
    (0.0,  4.0, 'D:maj'),
    (4.0,  8.0, 'F#:min'),
    (8.0, 12.0, 'E:maj'),
]

print("Chord at various times:")
for t in [0.5, 2.0, 4.0, 6.5, 10.0, 15.0]:
    chord = chord_at_time(t, fake_progression)
    print(f"  t={t}s: {chord}")

Chord at various times:
  t=0.5s: D:maj
  t=2.0s: D:maj
  t=4.0s: F#:min
  t=6.5s: F#:min
  t=10.0s: E:maj
  t=15.0s: None


In [ ]:
# ============================================================
# Cell 7: Music-Theory-Aware Fret Assignment (v1)
# ============================================================
# For each note, score every valid (string, fret) position
# based on music theory and playing context, then pick the best.

def score_position(
    midi: int,
    candidate: tuple[int, int],
    detected_key: tuple[str, str],
    current_chord: Optional[str],
    previous_position: Optional[tuple[int, int]],
    weights: Optional[dict] = None,
) -> float:
    """Score a candidate (string, fret) position. Higher = better."""
    if weights is None:
        weights = {
            'key_alignment':       1.0,
            'chord_tone':          2.0,
            'open_string_bonus':   1.0,
            'low_position_bonus':  0.5,    # frets 0-3 are "home"
            'middle_neck_bonus':   0.3,    # frets 4-12 are also comfortable
            'position_continuity': 0.5,    # softer than before
            'continuity_cap':      5.0,    # max fret distance that's penalized
        }

    string, fret = candidate
    score = 0.0

    tonic, mode = detected_key
    if is_in_key(midi, tonic, mode):
        score += weights['key_alignment']

    if current_chord is not None and is_chord_tone(midi, current_chord):
        score += weights['chord_tone']

    # Position comfort bonuses
    if fret == 0:
        score += weights['open_string_bonus']
    elif fret <= 3:
        score += weights['low_position_bonus']
    elif 4 <= fret <= 12:
        score += weights['middle_neck_bonus']

    # Position continuity — softer and capped, so big jumps are penalized
    # but not catastrophically. Square root grows slower than linear.
    if previous_position is not None:
        prev_fret = previous_position[1]
        if prev_fret > 0 and fret > 0:
            fret_distance = min(abs(fret - prev_fret), weights['continuity_cap'])
            score -= weights['position_continuity'] * (fret_distance ** 0.5)

    return score


def music_theory_assign(
    notes: list[dict],
    detected_key: tuple[str, str],
    chord_progression: list[tuple],
) -> list[TabNote]:
    """Music-theory-aware fret assignment.

    Args:
        notes: list of dicts with 'start', 'duration', 'midi' (from Basic Pitch)
        detected_key: (tonic, mode) tuple from key detection
        chord_progression: list of (start, end, label) from chord detection

    Returns:
        list of TabNote with assigned positions
    """
    tab_notes = []
    previous_position = None

    for n in notes:
        positions = midi_to_positions(n['midi'])
        if not positions:
            continue

        current_chord = chord_at_time(n['start'], chord_progression)

        # Score every candidate position
        scored = [
            (pos, score_position(n['midi'], pos, detected_key,
                                  current_chord, previous_position))
            for pos in positions
        ]
        # Pick highest-scoring
        best_pos, best_score = max(scored, key=lambda x: x[1])

        tab_notes.append(TabNote(
            start=n['start'],
            duration=n['duration'],
            midi=n['midi'],
            string=best_pos[0],
            fret=best_pos[1],
            confidence=best_score / 5.0,  # rough normalization for display
        ))
        previous_position = best_pos

    return tab_notes


# Smoke test — same notes as the naive version, but now with musical context
test_notes = [
    {'start': 0.0, 'duration': 0.5, 'midi': 64},  # E4
    {'start': 0.5, 'duration': 0.5, 'midi': 67},  # G4
    {'start': 1.0, 'duration': 0.5, 'midi': 71},  # B4
    {'start': 1.5, 'duration': 0.5, 'midi': 76},  # E5
]
test_key = ('E', 'minor')
test_chords = [(0.0, 2.0, 'E:min')]  # All notes happen during an Em chord

smart_tab = music_theory_assign(test_notes, test_key, test_chords)
print("Music-theory-aware assignment of E4 → G4 → B4 → E5 in E minor / Em chord:")
for tn in smart_tab:
    print(f"  {tn}  (confidence={tn.confidence:.2f})")

Music-theory-aware assignment of E4 → G4 → B4 → E5 in E minor / Em chord:
  TabNote(t=0.00s, E4, string=5, fret=0)  (confidence=0.80)
  TabNote(t=0.50s, G4, string=5, fret=3)  (confidence=0.70)
  TabNote(t=1.00s, B4, string=5, fret=7)  (confidence=0.46)
  TabNote(t=1.50s, E5, string=5, fret=12)  (confidence=0.44)


In [ ]:

available_recordings = sorted([
    p.stem.replace('_mic', '')
    for p in AUDIO_DIR.glob('*_mic.wav')
])
print(f"Found {len(available_recordings)} recordings:")
for r in available_recordings:
    print(f"  {r}")

Found 360 recordings:
  00_BN1-129-Eb_comp
  00_BN1-129-Eb_solo
  00_BN1-147-Gb_comp
  00_BN1-147-Gb_solo
  00_BN2-131-B_comp
  00_BN2-131-B_solo
  00_BN2-166-Ab_comp
  00_BN2-166-Ab_solo
  00_BN3-119-G_comp
  00_BN3-119-G_solo
  00_BN3-154-E_comp
  00_BN3-154-E_solo
  00_Funk1-114-Ab_comp
  00_Funk1-114-Ab_solo
  00_Funk1-97-C_comp
  00_Funk1-97-C_solo
  00_Funk2-108-Eb_comp
  00_Funk2-108-Eb_solo
  00_Funk2-119-G_comp
  00_Funk2-119-G_solo
  00_Funk3-112-C#_comp
  00_Funk3-112-C#_solo
  00_Funk3-98-A_comp
  00_Funk3-98-A_solo
  00_Jazz1-130-D_comp
  00_Jazz1-130-D_solo
  00_Jazz1-200-B_comp
  00_Jazz1-200-B_solo
  00_Jazz2-110-Bb_comp
  00_Jazz2-110-Bb_solo
  00_Jazz2-187-F#_comp
  00_Jazz2-187-F#_solo
  00_Jazz3-137-Eb_comp
  00_Jazz3-137-Eb_solo
  00_Jazz3-150-C_comp
  00_Jazz3-150-C_solo
  00_Rock1-130-A_comp
  00_Rock1-130-A_solo
  00_Rock1-90-C#_comp
  00_Rock1-90-C#_solo
  00_Rock2-142-D_comp
  00_Rock2-142-D_solo
  00_Rock2-85-F_comp
  00_Rock2-85-F_solo
  00_Rock3-117-Bb_comp

In [ ]:
# ============================================================
# Fret Assignment Eval — Experiment A
# ============================================================
# Run naive and music-theory-aware assignment on each recording's
# ground-truth MIDI; compare both to ground-truth (string, fret).
#
# Why feed GT MIDI instead of Basic Pitch output: this isolates the
# assigner from upstream detection error. The gap between naive and
# smart here is the cleanest defensible measure of what the
# music-theory layer is actually worth.

import numpy as np
import pandas as pd


def parse_jams_key(key_label):
    """Parse a JAMS key label ('Eb:major') into a (tonic, mode) tuple
    matching what music_theory_assign expects (('Eb', 'major')).

    Returns None if the label is missing or malformed. The fretboard
    algorithm's is_in_key expects mode == 'major' (anything else =
    minor), so we keep the full word, not the 'maj'/'min' abbreviation
    used for chord qualities elsewhere.
    """
    if not key_label:
        return None
    label = key_label.replace(':', ' ').strip()
    parts = label.split()
    if len(parts) < 2:
        return None
    tonic = parts[0]
    mode = parts[1].lower()
    # Accept abbreviated forms just in case
    mode = {'maj': 'major', 'min': 'minor'}.get(mode, mode)
    if mode not in ('major', 'minor'):
        return None
    return (tonic, mode)


def evaluate_fret_assignment(predicted_tabs, ground_truth_notes):
    """Compare predicted TabNotes to GuitarSet ground-truth notes 1:1.

    ground_truth_notes is recording['notes'] from the loader — flat
    list of dicts with keys: start, duration, string, fret, midi, note_name.

    Alignment is positional because both branches see the same inputs in
    the same order. For a fixed MIDI value, choosing a string uniquely
    determines the fret, so string accuracy == position accuracy here;
    we report position accuracy as the headline and mean fret distance
    as the soft metric (captures the magnitude of error when wrong).
    """
    assert len(predicted_tabs) == len(ground_truth_notes), (
        f"Alignment broken: {len(predicted_tabs)} predictions vs "
        f"{len(ground_truth_notes)} GT notes."
    )

    if not predicted_tabs:
        return {'n_notes': 0, 'position_accuracy': 0.0,
                'mean_fret_distance': 0.0, 'mean_fret_distance_wrong': 0.0}

    matches = 0
    all_dists = []
    wrong_dists = []

    for pred, gt in zip(predicted_tabs, ground_truth_notes):
        is_match = (pred.string == gt['string'] and pred.fret == gt['fret'])
        dist = abs(pred.fret - gt['fret'])
        all_dists.append(dist)
        if is_match:
            matches += 1
        else:
            wrong_dists.append(dist)

    n = len(predicted_tabs)
    return {
        'n_notes': n,
        'position_accuracy':        matches / n,
        'mean_fret_distance':       float(np.mean(all_dists)),
        'mean_fret_distance_wrong': float(np.mean(wrong_dists)) if wrong_dists else 0.0,
    }


def run_fret_assignment_batch(recording_ids, use_gt_context=True, verbose=True):
    """Run both assigners on each recording's ground-truth MIDI and
    compare to ground-truth fret positions.

    Args:
        recording_ids: list of GuitarSet stems (e.g. ['00_BN1-129-Eb_comp']).
        use_gt_context: if True (default), the smart assigner sees the
            ground-truth key and chords. This is Experiment A — the
            algorithm-only eval. Flip to False for the end-to-end version
            where smart consumes Basic Pitch / autochord / Krumhansl
            outputs (Experiment B, harder, compounds detection errors).
        verbose: print per-recording progress.

    Returns DataFrame, one row per recording.
    """
    rows = []
    for i, rid in enumerate(recording_ids):
        if verbose:
            print(f"[{i+1}/{len(recording_ids)}] {rid}")
        try:
            rec = load_guitarset_recording(rid)

            # Filter to notes the fretboard can actually represent. For
            # GuitarSet GT this should always be every note, but it's a
            # cheap safety net — if it ever fires, the 1:1 alignment
            # would silently break without it.
            gt_notes = [n for n in rec['notes'] if midi_to_positions(n['midi'])]
            skipped = len(rec['notes']) - len(gt_notes)
            if skipped and verbose:
                print(f"  ⚠ skipped {skipped} out-of-range notes")

            # Strip (string, fret) so the assigner has to rederive them.
            # Keep the original gt_notes for evaluation.
            input_notes = [
                {'start': n['start'], 'duration': n['duration'], 'midi': n['midi']}
                for n in gt_notes
            ]

            # Smart assigner needs key as (tonic, mode) and chords as
            # [(start, end, label)]. Use GT for Experiment A.
            if use_gt_context:
                key = parse_jams_key(rec['key'])
                chords = rec['chords']
            else:
                key_pred = run_key_detection(rec['audio_path'])
                key = parse_jams_key(key_pred['key'])
                chords = run_chord_detection(rec['audio_path'])

            if key is None:
                print(f"  ⚠ skipping — could not parse key {rec['key']!r}")
                continue

            naive_tabs = naive_assign(input_notes)
            smart_tabs = music_theory_assign(
                input_notes, detected_key=key, chord_progression=chords,
            )

            naive_m = evaluate_fret_assignment(naive_tabs, gt_notes)
            smart_m = evaluate_fret_assignment(smart_tabs, gt_notes)

            rows.append({
                'recording_id':    rid,
                'style':           rec['style'],
                'is_comp':         rec['is_comp'],
                'n_notes':         naive_m['n_notes'],
                'naive_pos_acc':   naive_m['position_accuracy'],
                'smart_pos_acc':   smart_m['position_accuracy'],
                'gap':             smart_m['position_accuracy'] - naive_m['position_accuracy'],
                'naive_fret_dist': naive_m['mean_fret_distance'],
                'smart_fret_dist': smart_m['mean_fret_distance'],
            })
        except Exception as e:
            print(f"  ❌ {rid}: {e}")

    return pd.DataFrame(rows)


def summarize_fret_results(df):
    """Print headline numbers."""
    if df.empty:
        print("No results to summarize.")
        return
    print("=" * 60)
    print(f"FRET ASSIGNMENT EVAL  "
          f"({len(df)} recordings, {df['n_notes'].sum()} total notes)")
    print("=" * 60)
    print(f"Naive position accuracy:    {df['naive_pos_acc'].mean():.3f}")
    print(f"Smart position accuracy:    {df['smart_pos_acc'].mean():.3f}")
    print(f"Gap (smart − naive):        {df['gap'].mean():+.3f}")
    print()
    print(f"Naive mean fret distance:   {df['naive_fret_dist'].mean():.2f}")
    print(f"Smart mean fret distance:   {df['smart_fret_dist'].mean():.2f}")

    if df['is_comp'].nunique() > 1:
        print("\nBY INPUT TYPE")
        by_type = df.groupby('is_comp')[
            ['naive_pos_acc', 'smart_pos_acc', 'gap']
        ].mean().round(3)
        by_type.index = ['Solo' if not c else 'Comp' for c in by_type.index]
        print(by_type.to_string())

    if df['style'].nunique() > 1:
        print("\nBY STYLE")
        by_style = df.groupby('style')[
            ['naive_pos_acc', 'smart_pos_acc', 'gap']
        ].mean().round(3)
        print(by_style.to_string())

In [ ]:
test_results = run_fret_assignment_batch(['00_BN1-129-Eb_comp'])
summarize_fret_results(test_results)

[1/1] 00_BN1-129-Eb_comp
FRET ASSIGNMENT EVAL  (1 recordings, 133 total notes)
Naive position accuracy:    0.105
Smart position accuracy:    0.271
Gap (smart − naive):        +0.165

Naive mean fret distance:   4.20
Smart mean fret distance:   3.45


In [ ]:
results = run_fret_assignment_batch(available_recordings)
summarize_fret_results(results)
results.sort_values('gap', ascending=False)

[1/360] 00_BN1-129-Eb_comp
[2/360] 00_BN1-129-Eb_solo
[3/360] 00_BN1-147-Gb_comp
[4/360] 00_BN1-147-Gb_solo
[5/360] 00_BN2-131-B_comp
[6/360] 00_BN2-131-B_solo
[7/360] 00_BN2-166-Ab_comp
[8/360] 00_BN2-166-Ab_solo
[9/360] 00_BN3-119-G_comp
[10/360] 00_BN3-119-G_solo
[11/360] 00_BN3-154-E_comp
[12/360] 00_BN3-154-E_solo
[13/360] 00_Funk1-114-Ab_comp
[14/360] 00_Funk1-114-Ab_solo
[15/360] 00_Funk1-97-C_comp
[16/360] 00_Funk1-97-C_solo
[17/360] 00_Funk2-108-Eb_comp
[18/360] 00_Funk2-108-Eb_solo
[19/360] 00_Funk2-119-G_comp
[20/360] 00_Funk2-119-G_solo
[21/360] 00_Funk3-112-C#_comp
[22/360] 00_Funk3-112-C#_solo
[23/360] 00_Funk3-98-A_comp
[24/360] 00_Funk3-98-A_solo
[25/360] 00_Jazz1-130-D_comp
[26/360] 00_Jazz1-130-D_solo
[27/360] 00_Jazz1-200-B_comp
[28/360] 00_Jazz1-200-B_solo
[29/360] 00_Jazz2-110-Bb_comp
[30/360] 00_Jazz2-110-Bb_solo
[31/360] 00_Jazz2-187-F#_comp
[32/360] 00_Jazz2-187-F#_solo
[33/360] 00_Jazz3-137-Eb_comp
[34/360] 00_Jazz3-137-Eb_solo
[35/360] 00_Jazz3-150-C_comp
[36/

,recording_id,style,is_comp,n_notes,naive_pos_acc,smart_pos_acc,gap,naive_fret_dist,smart_fret_dist
208,03_Jazz2-110-Bb_comp,Jazz2,True,132,0.000000,0.848485,0.848485,6.212121,0.825758
113,01_SS2-107-Ab_solo,SS2,False,243,0.041152,0.580247,0.539095,6.580247,3.300412
20,00_Funk3-112-C#_comp,Funk3,True,492,0.184959,0.717480,0.532520,5.615854,1.619919
131,02_BN3-154-E_solo,BN3,False,79,0.037975,0.569620,0.531646,6.518987,3.455696
93,01_Jazz3-137-Eb_solo,Jazz3,False,121,0.123967,0.636364,0.512397,5.157025,2.057851
...,...,...,...,...,...,...,...,...,...
9,00_BN3-119-G_solo,BN3,False,79,0.721519,0.405063,-0.316456,1.291139,2.822785
302,05_BN1-147-Gb_comp,BN1,True,96,0.656250,0.312500,-0.343750,1.552083,3.156250
243,04_BN1-147-Gb_solo,BN1,False,62,0.838710,0.451613,-0.387097,0.645161,2.580645
5,00_BN2-131-B_solo,BN2,False,97,0.391753,0.000000,-0.391753,4.567010,6.907216


In [ ]:
results.head(100)

,recording_id,style,is_comp,n_notes,naive_pos_acc,smart_pos_acc,gap,naive_fret_dist,smart_fret_dist
0,00_BN1-129-Eb_comp,BN1,True,133,0.105263,0.270677,0.165414,4.203008,3.451128
1,00_BN1-129-Eb_solo,BN1,False,71,0.507042,0.563380,0.056338,2.366197,2.098592
2,00_BN1-147-Gb_comp,BN1,True,151,0.953642,0.834437,-0.119205,0.185430,0.761589
3,00_BN1-147-Gb_solo,BN1,False,89,1.000000,1.000000,0.000000,0.000000,0.000000
4,00_BN2-131-B_comp,BN2,True,239,0.330544,0.276151,-0.054393,3.635983,3.878661
...,...,...,...,...,...,...,...,...,...
95,01_Jazz3-150-C_solo,Jazz3,False,109,0.137615,0.183486,0.045872,4.027523,3.798165
96,01_Rock1-130-A_comp,Rock1,True,196,1.000000,1.000000,0.000000,0.000000,0.000000
97,01_Rock1-130-A_solo,Rock1,False,87,0.034483,0.045977,0.011494,4.471264,4.425287
98,01_Rock1-90-C#_comp,Rock1,True,288,0.541667,0.506944,-0.034722,2.163194,2.340278


In [ ]:
# ============================================================
# Cell 3: Pipeline Wrappers
# ============================================================
# Thin wrappers around our three detection tools.
# Each function takes an audio path and returns predictions
# in a format that matches the corresponding ground truth field
# from load_guitarset_recording().

from basic_pitch.inference import predict as basic_pitch_predict


# ---------- Key Detection ----------

# Krumhansl-Schmuckler key profiles
_MAJOR_PROFILE = np.array([6.35, 2.23, 3.48, 2.33, 4.38, 4.09,
                            2.52, 5.19, 2.39, 3.66, 2.29, 2.88])
_MINOR_PROFILE = np.array([6.33, 2.68, 3.52, 5.38, 2.60, 3.53,
                            2.54, 4.75, 3.98, 2.69, 3.34, 3.17])
_PITCH_CLASSES = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']


def run_key_detection(audio_path):
    """
    Detect the key of an audio file using Krumhansl-Schmuckler on chromagram.

    Returns:
        dict with keys:
            - key: predicted key label, e.g. "D# major"
            - confidence_gap: gap between top-1 and top-2 (rough confidence)
            - ranked: top 5 candidates with correlation scores
    """
    y, sr = librosa.load(audio_path, sr=None)
    chroma = librosa.feature.chroma_cqt(y=y, sr=sr)
    chroma_avg = np.mean(chroma, axis=1)

    correlations = {}
    for tonic in range(12):
        major_template = np.roll(_MAJOR_PROFILE, tonic)
        minor_template = np.roll(_MINOR_PROFILE, tonic)
        correlations[f"{_PITCH_CLASSES[tonic]} major"] = np.corrcoef(chroma_avg, major_template)[0, 1]
        correlations[f"{_PITCH_CLASSES[tonic]} minor"] = np.corrcoef(chroma_avg, minor_template)[0, 1]

    ranked = sorted(correlations.items(), key=lambda x: x[1], reverse=True)
    top_key, top_score = ranked[0]
    _, second_score = ranked[1]

    return {
        'key': top_key,
        'confidence_gap': top_score - second_score,
        'ranked': ranked[:5],
    }


# ---------- Chord Detection ----------

def run_chord_detection(audio_path):
    """
    Detect chord progression using autochord.

    Returns:
        list of (start, end, label) tuples in the same shape as
        ground truth chord annotations from the loader.
    """
    chords = autochord.recognize(audio_path)
    # autochord already returns (start, end, label) tuples
    return [(start, end, label) for start, end, label in chords]


# ---------- Note Detection ----------

def run_note_detection(audio_path):
    """
    Detect notes using Basic Pitch.

    Returns:
        list of dicts with keys (start, duration, midi, note_name).
        Note: no string/fret info — that's our fretboard algorithm's job later.
    """
    _, _, note_events = basic_pitch_predict(audio_path)
    notes = []
    for start, end, pitch_midi, amplitude, _ in note_events:
        # Filter out low-amplitude artifacts and out-of-guitar-range notes
        if amplitude < 0.3 or pitch_midi < 40 or pitch_midi > 88:
            continue
        notes.append({
            'start': start,
            'duration': end - start,
            'midi': int(round(pitch_midi)),
            'note_name': _midi_to_note_name(pitch_midi),
            'amplitude': amplitude,
        })
    notes.sort(key=lambda n: n['start'])
    return notes


# ---------- Smoke Test ----------

test_id = "00_BN1-129-Eb_comp"
rec = load_guitarset_recording(test_id)

print(f"Running pipeline on: {rec['id']}\n")

print("Key detection...")
key_pred = run_key_detection(rec['audio_path'])
print(f"  Predicted: {key_pred['key']} (gap={key_pred['confidence_gap']:.3f})")
print(f"  Truth:     {rec['key']}\n")

print("Chord detection...")
chord_pred = run_chord_detection(rec['audio_path'])
print(f"  Predicted: {len(chord_pred)} segments")
print(f"  Truth:     {len(rec['chords'])} segments\n")

print("Note detection...")
note_pred = run_note_detection(rec['audio_path'])
print(f"  Predicted: {len(note_pred)} notes")
print(f"  Truth:     {len(rec['notes'])} per-string note annotations")

Running pipeline on: 00_BN1-129-Eb_comp

Key detection...
  Predicted: D# major (gap=0.091)
  Truth:     Eb:major

Chord detection...
1/1 [==============================] - 1s 852ms/step
  Predicted: 9 segments
  Truth:     6 segments

Note detection...
Predicting MIDI for /content/drive/MyDrive/Capstone/GuitarSet/Audio/00_BN1-129-Eb_comp_mic.wav...
  Predicted: 147 notes
  Truth:     133 per-string note annotations


In [ ]:
# ============================================================
# Run on Personal Recording (no ground truth)
# ============================================================
# Runs the full pipeline (key + chord + note detection) and then the
# music-theory-aware fret assigner on an arbitrary audio file.
# No GT here, so no accuracy numbers — just the predicted tab.
# Naive is shown alongside smart so you can see what the music theory
# layer is actually contributing.

from pathlib import Path

audio_path = '/content/drive/MyDrive/Capstone/Audio/New Recording 4.m4a'
audio_path = '/content/drive/MyDrive/Capstone/Audio/New Recording 4.m4a'


if not Path(audio_path).exists():
    raise FileNotFoundError(
        f"Not found: {audio_path}\n"
        "Double-check the path. The Capstone/Audio folder lives one level "
        "up from Capstone/GuitarSet/Audio."
    )

print(f"Loading: {audio_path}\n")

# --- Detection pipeline ---
print("Running key detection...")
key_pred = run_key_detection(audio_path)
print(f"  Detected key: {key_pred['key']} "
      f"(confidence gap: {key_pred['confidence_gap']:.3f})")

print("\nRunning chord detection...")
chords = run_chord_detection(audio_path)
print(f"  Detected {len(chords)} chord segments")

print("\nRunning note detection...")
notes = run_note_detection(audio_path)
print(f"  Detected {len(notes)} notes")

# --- Parse the key string into the (tonic, mode) tuple the assigner expects ---
key_tuple = parse_jams_key(key_pred['key'])
if key_tuple is None:
    raise ValueError(f"Could not parse detected key: {key_pred['key']}")

# --- Run both assigners ---
naive_tabs = naive_assign(notes)
smart_tabs = music_theory_assign(
    notes, detected_key=key_tuple, chord_progression=chords,
)

# --- Side-by-side output ---
print("\n" + "=" * 72)
print(f"PREDICTED TAB  |  key={key_pred['key']}  |  {len(smart_tabs)} notes")
print("=" * 72)
print(f"{'t':>6}  {'note':>5}  {'chord':>10}  "
      f"{'naive':>10}  {'smart':>10}")
print("-" * 72)

MAX_ROWS = 60   # bump or remove if you want the whole song
for i, (n, s) in enumerate(zip(naive_tabs, smart_tabs)):
    if i >= MAX_ROWS:
        print(f"  ... ({len(smart_tabs) - MAX_ROWS} more notes — "
              f"raise MAX_ROWS to see them)")
        break
    chord = chord_at_time(n.start, chords) or '-'
    naive_pos = f"s{n.string}f{n.fret}"
    smart_pos = f"s{s.string}f{s.fret}"
    diff = "  ←diff" if (n.string != s.string or n.fret != s.fret) else ""
    print(f"{n.start:6.2f}  {n.note_name:>5}  {chord:>10}  "
          f"{naive_pos:>10}  {smart_pos:>10}{diff}")

# Keep these around so you can poke at them after the cell finishes
print("\nAvailable in namespace: notes, chords, key_pred, naive_tabs, smart_tabs")

Loading: /content/drive/MyDrive/Capstone/Audio/New Recording 4.m4a

Running key detection...


/tmp/ipykernel_14502/2265350255.py:32: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(audio_path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


  Detected key: G major (confidence gap: 0.030)

Running chord detection...


/usr/local/lib/python3.12/dist-packages/autochord/__init__.py:93: UserWarning: PySoundFile failed. Trying audioread instead.
  samples, fs = librosa.load(audio_fn, sr=None, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


1/1 [==============================] - 0s 50ms/step
  Detected 5 chord segments

Running note detection...
Predicting MIDI for /content/drive/MyDrive/Capstone/Audio/New Recording 4.m4a...


/usr/local/lib/python3.12/dist-packages/basic_pitch/inference.py:229: UserWarning: PySoundFile failed. Trying audioread instead.
  audio_original, _ = librosa.load(str(audio_path), sr=AUDIO_SAMPLE_RATE, mono=True)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


  Detected 23 notes

PREDICTED TAB  |  key=G major  |  23 notes
     t   note       chord       naive       smart
------------------------------------------------------------------------
  1.07     E3       D:maj        s2f2        s2f2
  1.32     G3       D:maj        s3f0        s3f0
  1.65     A3       D:maj        s3f2        s3f2
  1.99     C4       D:maj        s4f1        s4f1
  2.32     A3       D:maj        s3f2        s3f2
  2.65     G4       D:maj        s5f3        s5f3
  3.07     D4       D:maj        s4f3        s4f3
  3.62     C4       D:maj        s4f1        s4f1
  3.93     A3       D:maj        s3f2        s3f2
  4.28     C4       D:maj        s4f1        s4f1
  4.98     D4       D:maj        s4f3        s4f3
  6.11     E3           N        s2f2        s2f2
  6.40     G3           N        s3f0        s3f0
  6.77     A3       D:maj        s3f2        s3f2
  7.09     C4       D:maj        s4f1        s4f1
  7.42     A3       D:maj        s3f2        s3f2
  7.77     G4